In [ ]:
# ============================================================
# CELL 1 — Install Dependencies
# ============================================================
import subprocess, sys

print('📦 Installing Unsloth and dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps',
    'bitsandbytes', 'accelerate', 'xformers==0.0.29.post3',
    'peft', 'trl', 'triton', 'cut_cross_entropy', 'unsloth_zoo'],
    check=True, capture_output=True)

subprocess.run([sys.executable, '-m', 'pip', 'install',
    'sentencepiece', 'protobuf', 'datasets>=3.4.1',
    'huggingface_hub', 'hf_transfer', 'tqdm'],
    check=True, capture_output=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', 'unsloth'],
    check=True, capture_output=True)

print('✅ Installation complete!')

📦 Installing Unsloth and dependencies...
✅ Installation complete!


In [ ]:

from unsloth import FastLanguageModel
import torch

MODEL_NAME  = 'unsloth/DeepSeek-R1-Distill-Qwen-7B-bnb-4bit'
MAX_SEQ_LEN = 2048

print(f'🤖 Loading model: {MODEL_NAME}')
print(f'💾 GPU : {torch.cuda.get_device_properties(0).name}')
print(f'💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
)

print('✅ Model loaded!')
print(f'📊 Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
🤖 Loading model: unsloth/DeepSeek-R1-Distill-Qwen-7B-bnb-4bit
💾 GPU : NVIDIA A100-SXM4-40GB
💾 VRAM: 42.4 GB
==((====))==  Unsloth 2026.5.5: Fast Qwen2 patching. Transformers: 5.0.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

Unsloth: Will load unsloth/DeepSeek-R1-Distill-Qwen-7B-bnb-4bit as a legacy tokenizer.


unsloth/DeepSeek-R1-Distill-Qwen-7B-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
✅ Model loaded!
📊 Parameters: 4.4B


In [ ]:

model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                      'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha     = 16,
    lora_dropout   = 0,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'✅ LoRA ready!')
print(f'🎯 Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

Unsloth 2026.5.5 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


✅ LoRA ready!
🎯 Trainable: 40,370,176 / 4,393,342,464 (0.92%)


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

!cp /content/drive/MyDrive/50k.jsonl /content/50k.jsonl
print('✅ Copied from Drive!')

import json
import pandas as pd

print('📥 Loading dataset...')
data    = []
skipped = 0

with open('/content/50k.jsonl', 'r') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            data.append(json.loads(line))
        except json.JSONDecodeError:
            skipped += 1
            continue

df = pd.DataFrame(data)

print(f'✅ Dataset loaded!')
print(f'   Total   : {len(df):,}')
print(f'   Phishing: {(df["label"] == "phishing").sum():,}')
print(f'   Benign  : {(df["label"] == "benign").sum():,}')
print(f'   Skipped : {skipped} malformed lines')
print(f'   Features: {len(df.iloc[0]["features"])} per example')

Mounted at /content/drive
✅ Copied from Drive!
📥 Loading dataset...
✅ Dataset loaded!
   Total   : 147,524
   Phishing: 59,600
   Benign  : 87,924
   Skipped : 0 malformed lines
   Features: 126 per example


In [ ]:
# ============================================================
# CELL 5 — Feature Grouping + Signal Generation + Reasoning
# ============================================================

def format_bool(val):
    return 'Yes' if val else 'No'

def format_ratio(val):
    return f'{val*100:.1f}%'


def build_feature_text(features):
    """Convert 126 numerical features into grouped readable text."""
    f = features

    text = f"""### URL ANALYSIS:
Total length: {f.get('url.final_url_length', 0)} | Hostname: {f.get('url.hostname_length', 0)} chars | Domain: {f.get('url.registrable_domain_length', 0)} chars
Subdomain: {f.get('url.subdomain_length', 0)} chars | Subdomain labels: {f.get('url.subdomain_label_count', 0)}
Path: {f.get('url.path_length', 0)} chars | Segments: {f.get('url.path_segment_count', 0)} | Query: {f.get('url.query_length', 0)} chars
HTTPS: {format_bool(f.get('url.scheme_is_https', 0))} | IP address: {format_bool(f.get('url.host_is_ip_address', 0))} | Punycode: {format_bool(f.get('url.punycode_present', 0))}
Digits: {format_ratio(f.get('url.digit_ratio', 0))} | Entropy: {f.get('url.character_entropy', 0):.2f}
Dots: {f.get('url.dot_count', 0)} | Hyphens: {f.get('url.hyphen_count', 0)} | Special chars: {f.get('url.special_character_count', 0)}
Tokens: {f.get('url.token_count', 0)} | Avg token length: {f.get('url.average_token_length', 0):.1f} | Longest token: {f.get('url.longest_token_length', 0)}

### PAGE STRUCTURE:
HTML length: {f.get('html.length', 0):,} | Visible text: {f.get('html.visible_text_length', 0):,} chars | Words: {f.get('html.visible_word_count', 0)}
Text/HTML ratio: {format_ratio(f.get('html.visible_text_to_html_ratio', 0))} | Text entropy: {f.get('html.visible_text_entropy', 0):.2f}
Title length: {f.get('html.title_length', 0)} | Domain token in title: {format_bool(f.get('html.current_domain_token_in_title', 0))}
Total tags: {f.get('html.total_tag_count', 0)} | Unique tags: {f.get('html.unique_tag_count', 0)}
Divs: {f.get('html.div_count', 0)} | Spans: {f.get('html.span_count', 0)} | Paragraphs: {f.get('html.paragraph_count', 0)}
Headings: {f.get('html.heading_h1_h2_h3_count', 0)} | Lists: {f.get('html.list_count', 0)} | Tables: {f.get('html.table_count', 0)}
Footer: {format_bool(f.get('html.footer_present', 0))} | Navigation: {format_bool(f.get('html.navigation_present', 0))} | Privacy/Terms link: {format_bool(f.get('html.privacy_or_terms_link_present', 0))}

### FORMS & INPUTS:
Forms: {f.get('html.form_count', 0)} | POST forms: {f.get('html.post_form_count', 0)}
Inputs: {f.get('html.input_count', 0)} | Text: {f.get('html.text_input_count', 0)} | Password: {f.get('html.password_input_count', 0)} | Email: {f.get('html.email_input_count', 0)}
Hidden inputs: {f.get('html.hidden_input_count', 0)} | Hidden ratio: {format_ratio(f.get('html.hidden_input_ratio', 0))}
Submit buttons: {f.get('html.submit_button_count', 0)} | Credential form: {format_bool(f.get('html.credential_form_present', 0))}
Null action forms: {f.get('html.null_form_action_count', 0)} | External action forms: {f.get('html.external_form_action_count', 0)}
Password form null action: {format_bool(f.get('html.password_form_null_action_present', 0))} | Password form external action: {format_bool(f.get('html.password_form_external_action_present', 0))}

### LINKS & ANCHORS:
Anchors: {f.get('html.anchors_with_href_count', 0)} | Internal: {f.get('html.internal_anchor_count', 0)} | External: {f.get('html.external_anchor_count', 0)}
Null/empty anchors: {f.get('html.null_or_empty_anchor_count', 0)} | Placeholder ratio: {format_ratio(f.get('html.placeholder_link_ratio', 0))}
External anchor ratio: {format_ratio(f.get('html.external_anchor_ratio', 0))}

### RESOURCES:
Images: {f.get('html.image_count', 0)} | External images: {f.get('html.external_image_count', 0)} | External image ratio: {format_ratio(f.get('html.external_image_ratio', 0))}
Scripts: {f.get('html.script_tag_count', 0)} | External scripts: {f.get('html.external_script_count', 0)} | Inline scripts: {f.get('html.inline_script_count', 0)}
Stylesheets: {f.get('html.stylesheet_link_count', 0)} | External stylesheets: {f.get('html.external_stylesheet_count', 0)}
Iframes: {f.get('html.iframe_count', 0)} | External iframes: {f.get('html.external_iframe_count', 0)}
Favicons: {f.get('html.favicon_count', 0)} | Total resource URLs: {f.get('html.resource_url_count', 0)}
Unique external domains: {f.get('html.unique_external_resource_domain_count', 0)} | External resource ratio: {format_ratio(f.get('html.external_resource_ratio', 0))}

### SECURITY INDICATORS:
Hidden element: {format_bool(f.get('html.hidden_element_present', 0))} | JS redirect: {format_bool(f.get('html.javascript_redirect_present', 0))}
eval() calls: {f.get('html.eval_call_count', 0)} | atob() calls: {f.get('html.atob_call_count', 0)} | document.write: {f.get('html.document_write_count', 0)}
Right-click disabled: {format_bool(f.get('html.right_click_disabling_present', 0))} | Alert/popup: {format_bool(f.get('html.alert_or_popup_present', 0))}
Meta refresh: {f.get('html.meta_refresh_count', 0)} | Meta tags: {f.get('html.meta_tag_count', 0)}

### REDIRECTS:
Redirect count: {f.get('metadata.redirect_count', 0)} | Domain changes: {f.get('metadata.redirect_domain_change_count', 0)}
URL changed: {format_bool(f.get('metadata.final_url_changed', 0))} | Host changed: {format_bool(f.get('metadata.final_host_changed', 0))} | Scheme changed: {format_bool(f.get('metadata.final_scheme_changed', 0))}"""

    return text


def build_signals(features):
    """Generate phishing and benign signal lists from numerical features."""
    f = features
    phishing_signals = []
    benign_signals   = []

    # Page structure
    if not f.get('html.navigation_present', 0):
        phishing_signals.append('⚠ No navigation menu detected — minimal page structure')
    else:
        benign_signals.append('✓ Navigation menu present')

    if not f.get('html.footer_present', 0):
        phishing_signals.append('⚠ No footer section detected')
    else:
        benign_signals.append('✓ Footer section present')

    if not f.get('html.privacy_or_terms_link_present', 0):
        phishing_signals.append('⚠ No privacy policy or terms of service links')
    else:
        benign_signals.append('✓ Privacy/terms links present')

    # URL
    if not f.get('url.scheme_is_https', 0):
        phishing_signals.append('⚠ HTTP scheme — unencrypted connection')
    else:
        benign_signals.append('✓ HTTPS used')

    if f.get('url.host_is_ip_address', 0):
        phishing_signals.append('⚠ Raw IP address used instead of domain name')

    if f.get('url.subdomain_length', 0) > 15:
        phishing_signals.append(f'⚠ Unusually long subdomain ({f["url.subdomain_length"]} chars)')

    if f.get('url.digit_ratio', 0) > 0.15:
        phishing_signals.append(f'⚠ High digit ratio in URL ({format_ratio(f["url.digit_ratio"])})')

    if f.get('url.punycode_present', 0):
        phishing_signals.append('⚠ Punycode encoding detected — potential homograph attack')

    if f.get('url.character_entropy', 0) > 4.5:
        phishing_signals.append(f'⚠ High URL entropy ({f["url.character_entropy"]:.2f}) — randomized domain')
    elif f.get('url.character_entropy', 0) < 3.5 and f.get('url.character_entropy', 0) > 0:
        benign_signals.append(f'✓ Normal URL entropy ({f["url.character_entropy"]:.2f})')

    # Forms & credentials
    if f.get('html.password_input_count', 0) > 0:
        phishing_signals.append(f'⚠ Password input field present ({f["html.password_input_count"]} field(s))')

    if f.get('html.credential_form_present', 0):
        phishing_signals.append('⚠ Credential harvesting form detected')

    if f.get('html.password_form_null_action_present', 0):
        phishing_signals.append('⚠ Password form has null action — likely JS-based credential theft')

    if f.get('html.password_form_external_action_present', 0):
        phishing_signals.append('⚠ Password form submits to external domain')

    if f.get('html.hidden_input_count', 0) > 3:
        phishing_signals.append(f'⚠ Many hidden input fields ({f["html.hidden_input_count"]}) — campaign tracking suspected')

    if f.get('html.null_form_action_count', 0) > 0:
        phishing_signals.append(f'⚠ Form with null/blank action detected')

    # Links
    if f.get('html.placeholder_link_ratio', 0) > 0.7:
        phishing_signals.append(f'⚠ Most links are placeholder/null ({format_ratio(f["html.placeholder_link_ratio"])})')
    elif f.get('html.placeholder_link_ratio', 0) < 0.2 and f.get('html.anchors_with_href_count', 0) > 5:
        benign_signals.append('✓ Page has well-structured navigation links')

    if f.get('html.internal_anchor_count', 0) > 10:
        benign_signals.append(f'✓ Rich internal navigation ({f["html.internal_anchor_count"]} internal links)')

    # Security
    if f.get('html.right_click_disabling_present', 0):
        phishing_signals.append('⚠ Right-click disabled — source inspection blocked')

    if f.get('html.javascript_redirect_present', 0):
        phishing_signals.append('⚠ JavaScript-based redirect detected')

    if f.get('html.eval_call_count', 0) > 0:
        phishing_signals.append(f'⚠ eval() calls detected ({f["html.eval_call_count"]}) — obfuscated code')

    if f.get('html.atob_call_count', 0) > 0:
        phishing_signals.append(f'⚠ atob() calls detected — base64 decoded execution')

    # Redirects
    if f.get('metadata.redirect_count', 0) > 2:
        phishing_signals.append(f'⚠ Multiple redirects ({f["metadata.redirect_count"]}) — redirect chain')

    if f.get('metadata.redirect_domain_change_count', 0) > 0:
        phishing_signals.append(f'⚠ Domain changed during redirect ({f["metadata.redirect_domain_change_count"]} time(s))')

    # Page content richness (benign indicators)
    if f.get('html.heading_h1_h2_h3_count', 0) > 3:
        benign_signals.append(f'✓ Well-structured content with {f["html.heading_h1_h2_h3_count"]} headings')

    if f.get('html.list_count', 0) > 5:
        benign_signals.append(f'✓ Rich content structure ({f["html.list_count"]} lists)')

    if f.get('html.visible_word_count', 0) > 500:
        benign_signals.append(f'✓ Substantial visible content ({f["html.visible_word_count"]} words)')

    if f.get('html.unique_tag_count', 0) > 25:
        benign_signals.append(f'✓ Complex page structure ({f["html.unique_tag_count"]} unique HTML tags)')

    return phishing_signals, benign_signals


def build_reasoning(features, label, phishing_signals=None, benign_signals=None):
    """Generate analyst reasoning referencing the detected signals."""
    f = features
    if phishing_signals is None or benign_signals is None:
        phishing_signals, benign_signals = build_signals(features)

    parts = []

    if label == 'phishing':
        if any('navigation' in s.lower() for s in phishing_signals):
            parts.append('The page lacks navigation elements typical of legitimate websites')
        if any('footer' in s.lower() for s in phishing_signals):
            parts.append('no footer section is present')
        if any('subdomain' in s.lower() for s in phishing_signals):
            parts.append('the URL uses an unusually long subdomain common in phishing')
        if any('password' in s.lower() or 'credential' in s.lower() for s in phishing_signals):
            parts.append('credential harvesting form with password field is present')
        if any('null action' in s.lower() for s in phishing_signals):
            parts.append('the password form has a null action suggesting JavaScript-based credential theft')
        if any('placeholder' in s.lower() for s in phishing_signals):
            parts.append('most links are placeholder links typical of phishing kits')
        if any('ip address' in s.lower() for s in phishing_signals):
            parts.append('a raw IP address is used instead of a domain name')
        if any('right-click' in s.lower() for s in phishing_signals):
            parts.append('right-click is disabled to prevent source inspection')
        if any('redirect' in s.lower() for s in phishing_signals):
            parts.append('suspicious redirect behavior was detected')
        if not parts:
            parts.append(f'Multiple structural indicators suggest this is not a legitimate website ({len(phishing_signals)} suspicious signals)')

        reasoning = '. '.join(parts[:4]).capitalize() + '.'
        if benign_signals:
            reasoning += f' While {len(benign_signals)} benign signal(s) are present, these do not outweigh the phishing indicators.'
        reasoning += ' Overall assessment: high-confidence phishing.'
    else:
        if any('navigation' in s.lower() for s in benign_signals):
            parts.append('The page has proper navigation elements')
        if any('footer' in s.lower() for s in benign_signals):
            parts.append('a footer section is present')
        if any('privacy' in s.lower() or 'terms' in s.lower() for s in benign_signals):
            parts.append('privacy policy and terms links are available')
        if any('heading' in s.lower() for s in benign_signals):
            parts.append('the page has well-structured content with proper headings')
        if any('https' in s.lower() for s in benign_signals):
            parts.append('HTTPS is used')
        if any('content' in s.lower() or 'word' in s.lower() for s in benign_signals):
            parts.append('the page contains substantial legitimate content')
        if not parts:
            parts.append('No significant phishing indicators were detected')

        reasoning = '. '.join(parts[:4]).capitalize() + '.'
        if phishing_signals:
            reasoning += f' While {len(phishing_signals)} minor suspicious signal(s) noted, these are insufficient to indicate phishing.'
        reasoning += ' Overall assessment: likely benign.'

    return reasoning


# Test
row = df.iloc[0]
phishing_s, benign_s = build_signals(row['features'])
reasoning = build_reasoning(row['features'], row['label'], phishing_s, benign_s)

print('✅ Signal + reasoning generation ready!')
print(f'\nLabel: {row["label"]}')
print(f'Phishing signals: {len(phishing_s)}')
print(f'Benign signals  : {len(benign_s)}')
print(f'\nSample phishing signals:')
for s in phishing_s[:3]:
    print(f'  {s}')
print(f'\nSample benign signals:')
for s in benign_s[:3]:
    print(f'  {s}')
print(f'\nReasoning: {reasoning}')

✅ Signal + reasoning generation ready!

Label: phishing
Phishing signals: 4
Benign signals  : 1

Sample phishing signals:
  ⚠ No navigation menu detected — minimal page structure
  ⚠ No footer section detected
  ⚠ No privacy policy or terms of service links

Sample benign signals:
  ✓ HTTPS used

Reasoning: The page lacks navigation elements typical of legitimate websites. no footer section is present. most links are placeholder links typical of phishing kits. While 1 benign signal(s) are present, these do not outweigh the phishing indicators. Overall assessment: high-confidence phishing.


In [ ]:
# ============================================================
# CELL 6 — Prompt Template
# ============================================================
# INPUT:  URL + title + grouped numerical features
# OUTPUT: phishing signals + benign signals + reasoning + verdict

from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template='qwen-2.5')

SYSTEM_PROMPT = """You are CyberGuard, an expert cybersecurity analyst specializing in phishing detection.

You will receive structured web page data including URL characteristics, page structure, forms, links, resources, security indicators, and redirect behavior.

Your task:
1. List PHISHING SIGNALS (⚠) — suspicious indicators found in the data
2. List BENIGN SIGNALS (✓) — legitimate indicators found in the data
3. Write a REASONING paragraph weighing all signals together
4. Give a final VERDICT

Important rules:
- Do NOT base your verdict on a single signal alone
- The absence of navigation or footer alone is NOT sufficient for a phishing verdict
- Always consider URL structure, form behavior, and redirect patterns together
- A page with password fields but strong benign signals may still be legitimate

Output format (strictly follow this):
### PHISHING SIGNALS:
⚠ [signal description]

### BENIGN SIGNALS:
✓ [signal description]

### REASONING:
[2-3 sentence analysis]

### VERDICT: PHISHING or VERDICT: BENIGN"""


def build_user_prompt(row):
    """Build input prompt with URL + title + grouped features."""
    url      = row.get('url', row.get('final_url', 'N/A'))
    title    = str(row.get('title', '') or '').strip()[:100]
    features = row.get('features', {})

    feature_text = build_feature_text(features)

    return f"""### TARGET:
URL: {url}
Page Title: {title if title else 'N/A'}

{feature_text}"""


def build_assistant_response(row):
    """Build expected output with signals + reasoning + verdict."""
    features = row.get('features', {})
    phishing_signals, benign_signals = build_signals(features)
    reasoning = build_reasoning(features, row['label'], phishing_signals, benign_signals)
    verdict   = 'PHISHING' if row['label'] == 'phishing' else 'BENIGN'

    phishing_text = '\n'.join(phishing_signals) if phishing_signals else '— None detected'
    benign_text   = '\n'.join(benign_signals)   if benign_signals   else '— None detected'

    return f"""### PHISHING SIGNALS:
{phishing_text}

### BENIGN SIGNALS:
{benign_text}

### REASONING:
{reasoning}

### VERDICT: {verdict}"""


EOS_TOKEN = tokenizer.eos_token

def row_to_text(row):
    messages = [
        {'role': 'system',    'content': SYSTEM_PROMPT},
        {'role': 'user',      'content': build_user_prompt(row)},
        {'role': 'assistant', 'content': build_assistant_response(row)},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    ) + EOS_TOKEN


# Preview
row = df.iloc[0]
print('--- USER PROMPT ---')
print(build_user_prompt(row))
print('\n--- ASSISTANT RESPONSE ---')
print(build_assistant_response(row))

# Token length check
sample_text = row_to_text(row)
tokens = tokenizer(sample_text, return_tensors='pt')
print(f'\n📊 Sample token length: {tokens["input_ids"].shape[1]}')

--- USER PROMPT ---
### TARGET:
URL: https://masukomu.onamaeweb.jp/max.il/activation/total/
Page Title: MAX - אימות כרטיס

### URL ANALYSIS:
Total length: 54 | Hostname: 21 chars | Domain: 12 chars
Subdomain: 8 chars | Subdomain labels: 1
Path: 25 chars | Segments: 3 | Query: 0 chars
HTTPS: Yes | IP address: No | Punycode: No
Digits: 0.0% | Entropy: 4.12
Dots: 3 | Hyphens: 0 | Special chars: 10
Tokens: 8 | Avg token length: 5.5 | Longest token: 10

### PAGE STRUCTURE:
HTML length: 20,541 | Visible text: 612 chars | Words: 118
Text/HTML ratio: 3.0% | Text entropy: 4.66
Title length: 17 | Domain token in title: No
Total tags: 69 | Unique tags: 17
Divs: 10 | Spans: 8 | Paragraphs: 0
Headings: 0 | Lists: 1 | Tables: 0
Footer: No | Navigation: No | Privacy/Terms link: No

### FORMS & INPUTS:
Forms: 1 | POST forms: 1
Inputs: 6 | Text: 6 | Password: 0 | Email: 0
Hidden inputs: 0 | Hidden ratio: 0.0%
Submit buttons: 2 | Credential form: No
Null action forms: 0 | External action forms: 0
Passwo

In [ ]:

from datasets import Dataset
from tqdm import tqdm

SAMPLE_SIZE = 16000
print(f'📊 Sampling {SAMPLE_SIZE} balanced examples...')
phishing_df = df[df['label'] == 'phishing'].sample(n=SAMPLE_SIZE // 2, random_state=42)
benign_df   = df[df['label'] == 'benign'].sample(n=SAMPLE_SIZE // 2, random_state=42)
sample_df   = pd.concat([phishing_df, benign_df]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'   Phishing: {len(phishing_df):,}')
print(f'   Benign  : {len(benign_df):,}')

print('\n⚙️  Converting to prompt format...')
texts = []
for _, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
    texts.append(row_to_text(row))

# 70-15-15 split
hf_dataset = Dataset.from_dict({'text': texts})
split_1    = hf_dataset.train_test_split(test_size=0.3, seed=42)   # 70 / 30
train_data = split_1['train']                                       # 70%
split_2    = split_1['test'].train_test_split(test_size=0.5, seed=42)  # 15 / 15
val_data   = split_2['train']                                       # 15%
test_data  = split_2['test']                                        # 15%

print(f'\n✅ Dataset ready (70-15-15 split)!')
print(f'   Train: {len(train_data):,}')
print(f'   Val  : {len(val_data):,}  (used for early stopping)')
print(f'   Test : {len(test_data):,}  (used for final evaluation)')

📊 Sampling 16000 balanced examples...
   Phishing: 8,000
   Benign  : 8,000

⚙️  Converting to prompt format...


100%|██████████| 16000/16000 [00:03<00:00, 4345.96it/s]



✅ Dataset ready (70-15-15 split)!
   Train: 11,200
   Val  : 2,400  (used for early stopping)
   Test : 2,400  (used for final evaluation)


In [ ]:

from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback
import torch, gc

torch.cuda.empty_cache()
gc.collect()

trainer = SFTTrainer(
    model            = model,
    processing_class = tokenizer,
    train_dataset    = train_data,
    eval_dataset     = val_data,
    args = SFTConfig(
        dataset_text_field          = 'text',
        max_seq_length              = None,
        packing                     = True,
        dataset_num_proc            = 2,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 16,
        per_device_eval_batch_size  = 1,
        num_train_epochs            = 5,
        learning_rate               = 2e-4,
        warmup_steps                = 50,
        lr_scheduler_type           = 'cosine',
        eval_strategy               = 'steps',
        eval_steps                  = 50,
        save_strategy               = 'steps',
        save_steps                  = 50,
        load_best_model_at_end      = True,
        metric_for_best_model       = 'eval_loss',
        greater_is_better           = False,
        save_total_limit            = 3,
        fp16                        = False,
        bf16                        = True,
        optim                       = 'adamw_8bit',
        weight_decay                = 0.01,
        logging_steps               = 20,
        gradient_checkpointing      = True,
        output_dir                  = '/content/deepseek-phishing-v3-checkpoints',
        report_to                   = 'none',
        seed                        = 42,
    ),
    callbacks = [EarlyStoppingCallback(early_stopping_patience=3)],
)

used_mem  = round(torch.cuda.max_memory_reserved() / 1024**3, 1)
total_mem = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1)
print(f'💾 GPU Memory: {used_mem}GB / {total_mem}GB in use')
print(f'\n🚀 Training with early stopping...')
print(f'   Max epochs: 5 | Patience: 3 evals (150 steps)')

trainer_stats = trainer.train()

print(f'\n✅ Training complete!')
print(f'   Duration  : {trainer_stats.metrics["train_runtime"]/60:.1f} minutes')
print(f'   Train loss: {trainer_stats.metrics["train_loss"]:.4f}')

print('\n📊 Eval loss history:')
for log in trainer.state.log_history:
    if 'eval_loss' in log:
        print(f'   Step {log["step"]:5d}: eval_loss={log["eval_loss"]:.4f}')

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/11200 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=2):   0%|          | 0/11200 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2400 [00:00<?, ? examples/s]

Unsloth: Packing eval dataset (num_proc=2):   0%|          | 0/2400 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!
💾 GPU Memory: 5.4GB / 39.5GB in use

🚀 Training with early stopping...
   Max epochs: 5 | Patience: 3 evals (150 steps)


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8,472 | Num Epochs = 5 | Total steps = 2,650
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
50,1.229807,0.350826
100,0.273030,0.269859
150,0.252747,0.256393
200,0.245313,0.248881
250,0.252466,0.240469
300,0.227403,0.233381
350,0.239869,0.227373
400,0.238109,0.223776
450,0.216002,0.221449
500,0.216123,0.218529


Unsloth: Restored added_tokens_decoder metadata in /content/deepseek-phishing-v3-checkpoints/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/deepseek-phishing-v3-checkpoints/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/deepseek-phishing-v3-checkpoints/checkpoint-150/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/deepseek-phishing-v3-checkpoints/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/deepseek-phishing-v3-checkpoints/checkpoint-250/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/deepseek-phishing-v3-checkpoints/checkpoint-300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/deepseek-phishing-v3-checkpoints/checkpoint-350/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/deepseek-phishing-v3-che


✅ Training complete!
   Duration  : 592.5 minutes
   Train loss: 0.2483

📊 Eval loss history:
   Step    50: eval_loss=0.3508
   Step   100: eval_loss=0.2699
   Step   150: eval_loss=0.2564
   Step   200: eval_loss=0.2489
   Step   250: eval_loss=0.2405
   Step   300: eval_loss=0.2334
   Step   350: eval_loss=0.2274
   Step   400: eval_loss=0.2238
   Step   450: eval_loss=0.2214
   Step   500: eval_loss=0.2185
   Step   550: eval_loss=0.2171
   Step   600: eval_loss=0.2156
   Step   650: eval_loss=0.2136
   Step   700: eval_loss=0.2115
   Step   750: eval_loss=0.2097
   Step   800: eval_loss=0.2091
   Step   850: eval_loss=0.2073
   Step   900: eval_loss=0.2064
   Step   950: eval_loss=0.2052
   Step  1000: eval_loss=0.2045
   Step  1050: eval_loss=0.2030
   Step  1100: eval_loss=0.2033
   Step  1150: eval_loss=0.2025
   Step  1200: eval_loss=0.2013
   Step  1250: eval_loss=0.2012
   Step  1300: eval_loss=0.2004
   Step  1350: eval_loss=0.2005
   Step  1400: eval_loss=0.1991
   Step  

In [ ]:

FastLanguageModel.for_inference(model)

def analyze(url, title, features):
    """Analyze a URL with its features."""
    row = {'url': url, 'title': title, 'features': features}
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': build_user_prompt(row)},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize              = True,
        add_generation_prompt = True,
        return_tensors        = 'pt',
        return_dict           = True,
    ).to('cuda')

    input_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            input_ids      = inputs['input_ids'],
            attention_mask = inputs['attention_mask'],
            max_new_tokens = 300,
            temperature    = 0.1,
            do_sample      = True,
            pad_token_id   = tokenizer.eos_token_id,
        )

    generated = outputs[0][input_len:]
    return tokenizer.decode(generated, skip_special_tokens=True)


for i, idx in enumerate([0, 1, 100]):
    row = df.iloc[idx]
    print('='*60)
    print(f'TEST {i+1} — True label: {row["label"].upper()}')
    print(f'URL: {row["url"][:80]}')
    print('='*60)
    result = analyze(
        url      = row['url'],
        title    = str(row.get('title', '') or ''),
        features = row['features'],
    )
    print(result)
    print()

TEST 1 — True label: PHISHING
URL: https://masukomu.onamaeweb.jp/max.il/activation/total/
### PHISHING SIGNALS:
⚠ No navigation menu detected — minimal page structure
⚠ No footer section detected
⚠ No privacy policy or terms terms of service links
⚠ Most links are placeholder/null (100.0%)

### BENIGN SIGNALS:
✓ HTTPS used

### REASONING:
The page lacks navigation elements typical of legitimate websites. no footer section is present. most links are placeholder links typical of phishing kits. While 1 benign signal(s) are present, these do not outweigh the phishing indicators. Overall assessment: high-confidence phishing.

### VERDICT: PHISHING<|im_end|>


TEST 2 — True label: PHISHING
URL: http://veruiozeruinmertfun.weebly.com/
### PHISHING SIGNALS:
⚠ No navigation menu detected — minimal page structure
⚠ No footer section detected
⚠ No privacy policy or terms terms links
⚠ Unusually long subdomain (19 chars)
⚠ Many hidden input fields (5) — campaign tracking suspected

### BENIGN SIGNA

In [ ]:
# ============================================================
# CELL 10 — Evaluation: Verdict + Explainability + Error Analysis
# ============================================================
# Uses TEST SET (separate from val used for early stopping)
# Explainability: checks if model's output signals match ground truth signals

import numpy as np
from tqdm import tqdm
from collections import Counter
import re


def predict_label(row, max_new_tokens=400):
    """Predict from features. Returns (verdict, full_output)."""
    user_row = {
        'url'      : row.get('url', row.get('final_url', '')),
        'title'    : str(row.get('title', '') or ''),
        'features' : row.get('features', {}),
    }
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': build_user_prompt(user_row)},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize              = True,
        add_generation_prompt = True,
        return_tensors        = 'pt',
        return_dict           = True,
    ).to('cuda')

    input_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            input_ids      = inputs['input_ids'],
            attention_mask = inputs['attention_mask'],
            max_new_tokens = max_new_tokens,
            temperature    = 0.1,
            do_sample      = True,
            pad_token_id   = tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(
        outputs[0][input_len:], skip_special_tokens=True
    )
    upper = generated.upper()

    if 'VERDICT: PHISHING' in upper:
        return 1, generated
    elif 'VERDICT: BENIGN' in upper:
        return 0, generated
    return -1, generated


# ============================================================
# Signal-based Explainability Scoring
# ============================================================
# Ground truth signals are computed from features (build_signals)
# Model output signals are extracted from the generated text
# Coverage = how many ground truth signals appear in model output

def extract_model_signals(generated_text):
    """Extract phishing and benign signals from model output."""
    model_phishing = []
    model_benign   = []

    lines = generated_text.split('\n')
    current_section = None

    for line in lines:
        line_upper = line.upper().strip()
        if 'PHISHING SIGNALS' in line_upper:
            current_section = 'phishing'
        elif 'BENIGN SIGNALS' in line_upper:
            current_section = 'benign'
        elif 'REASONING' in line_upper or 'VERDICT' in line_upper:
            current_section = None
        elif line.strip().startswith('⚠') and current_section == 'phishing':
            model_phishing.append(line.strip())
        elif line.strip().startswith('✓') and current_section == 'benign':
            model_benign.append(line.strip())

    return model_phishing, model_benign


def score_explainability(generated_text, features):
    """
    Compare model's output signals with ground truth signals.
    Coverage = % of ground truth signal categories found in model output.
    """
    # Ground truth signals
    gt_phishing, gt_benign = build_signals(features)

    if not gt_phishing and not gt_benign:
        return None

    # Model's output signals
    model_phishing, model_benign = extract_model_signals(generated_text)

    # Define category keywords to match signals
    SIGNAL_CATEGORIES = {
        'navigation'   : ['navigation', 'nav', 'menu'],
        'footer'       : ['footer'],
        'privacy'      : ['privacy', 'terms'],
        'https'        : ['https', 'http', 'ssl', 'unencrypted'],
        'ip_address'   : ['ip address', 'raw ip'],
        'subdomain'    : ['subdomain', 'long subdomain'],
        'digit'        : ['digit ratio', 'high digit'],
        'punycode'     : ['punycode', 'homograph'],
        'entropy'      : ['entropy'],
        'password'     : ['password', 'credential', 'credential form'],
        'null_action'  : ['null action', 'blank action'],
        'external_form': ['external domain', 'external action'],
        'hidden_input' : ['hidden input', 'campaign tracking'],
        'placeholder'  : ['placeholder', 'null link'],
        'right_click'  : ['right-click', 'right click'],
        'js_redirect'  : ['javascript redirect', 'js redirect'],
        'eval'         : ['eval()', 'obfuscated'],
        'redirect'     : ['redirect chain', 'multiple redirect'],
        'headings'     : ['heading', 'well-structured content'],
        'content'      : ['visible content', 'word count', 'substantial'],
        'internal_nav' : ['internal navigation', 'internal links'],
    }

    def signal_to_category(signal_text):
        """Map a signal text to its category."""
        sl = signal_text.lower()
        for cat, keywords in SIGNAL_CATEGORIES.items():
            if any(kw in sl for kw in keywords):
                return cat
        return None

    # Get ground truth categories
    gt_categories = set()
    for s in gt_phishing + gt_benign:
        cat = signal_to_category(s)
        if cat:
            gt_categories.add(cat)

    if not gt_categories:
        return None

    # Get model output categories
    model_categories = set()
    for s in model_phishing + model_benign:
        cat = signal_to_category(s)
        if cat:
            model_categories.add(cat)

    # Coverage = intersection / ground truth
    matched = len(gt_categories & model_categories)
    coverage = matched / len(gt_categories)

    return coverage


# ============================================================
# Run Evaluation on TEST SET
# ============================================================
train_size = len(train_data)
val_size   = len(val_data)
test_start = train_size + val_size
test_df    = sample_df.iloc[test_start:test_start + len(test_data)].reset_index(drop=True)

# Balance
test_phishing = test_df[test_df['label'] == 'phishing']
test_benign   = test_df[test_df['label'] == 'benign']
min_count     = min(len(test_phishing), len(test_benign))
test_phishing = test_phishing.head(min_count)
test_benign   = test_benign.head(min_count)
eval_df       = pd.concat([test_phishing, test_benign]).sample(frac=1, random_state=42).reset_index(drop=True)

TOTAL_EVAL = len(eval_df)

preds, labels         = [], []
explainability_scores = []
errors                = []
ambiguous_examples    = []

print(f'📊 Evaluating on TEST SET ({TOTAL_EVAL} examples, balanced)...')
print(f'   (This set was NOT used for training or early stopping)')

for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    pred, output = predict_label(row)
    true_label   = 1 if row['label'] == 'phishing' else 0
    features     = row.get('features', {})

    if pred != -1:
        preds.append(pred)
        labels.append(true_label)

        score = score_explainability(output, features)
        if score is not None:
            explainability_scores.append(score)

        if pred != true_label:
            errors.append({
                'url'      : row.get('url', ''),
                'true'     : 'phishing' if true_label == 1 else 'benign',
                'predicted': 'phishing' if pred == 1 else 'benign',
                'features' : features,
                'output'   : output,
            })
    else:
        ambiguous_examples.append({
            'url'        : row.get('url', ''),
            'true_label' : row['label'],
            'output'     : output,
        })

preds  = np.array(preds, dtype=int)
labels = np.array(labels, dtype=int)


# ============================================================
# VERDICT METRICS
# ============================================================
if len(preds) == 0:
    print('⚠️ No predictions — all ambiguous!')
else:
    accuracy  = (preds == labels).mean()
    tp = ((preds == 1) & (labels == 1)).sum()
    fp = ((preds == 1) & (labels == 0)).sum()
    fn = ((preds == 0) & (labels == 1)).sum()
    tn = ((preds == 0) & (labels == 0)).sum()
    precision = tp / (tp + fp + 1e-10)
    recall    = tp / (tp + fn + 1e-10)
    f1        = 2 * precision * recall / (precision + recall + 1e-10)

    print(f'\n📈 VERDICT METRICS (held-out test set):')
    print(f'   Accuracy : {accuracy:.4f} ({accuracy*100:.1f}%)')
    print(f'   Precision: {precision:.4f}')
    print(f'   Recall   : {recall:.4f}')
    print(f'   F1 Score : {f1:.4f}')
    print(f'\n   Confusion Matrix:')
    print(f'   TP={tp}  FP={fp}')
    print(f'   FN={fn}  TN={tn}')
    print(f'   Ambiguous: {TOTAL_EVAL - len(preds)}')


# ============================================================
# EXPLAINABILITY METRICS
# ============================================================
if explainability_scores:
    avg_explain    = np.mean(explainability_scores)
    median_explain = np.median(explainability_scores)
    perfect = sum(1 for s in explainability_scores if s == 1.0)
    zero    = sum(1 for s in explainability_scores if s == 0.0)

    print(f'\n💡 EXPLAINABILITY METRICS:')
    print(f'   (Signal coverage = % of ground truth signal categories reproduced in model output)')
    print(f'   Mean coverage   : {avg_explain:.4f} ({avg_explain*100:.1f}%)')
    print(f'   Median coverage : {median_explain:.4f}')
    print(f'   Perfect (100%)  : {perfect} / {len(explainability_scores)}')
    print(f'   Missed all (0%) : {zero} / {len(explainability_scores)}')


# ============================================================
# ERROR ANALYSIS
# ============================================================
print(f'\n🔍 ERROR ANALYSIS:')
print(f'   Total errors: {len(errors)}')

fp_errors = [e for e in errors if e['predicted'] == 'phishing']
fn_errors = [e for e in errors if e['predicted'] == 'benign']

print(f'   FP (benign → phishing): {len(fp_errors)}')
print(f'   FN (phishing → benign): {len(fn_errors)}')


def summarize_error_features(error_list, label):
    if not error_list:
        return
    print(f'\n   {label} error patterns:')
    nav    = sum(1 for e in error_list if not e['features'].get('html.navigation_present', 0))
    footer = sum(1 for e in error_list if not e['features'].get('html.footer_present', 0))
    passwd = sum(1 for e in error_list if e['features'].get('html.password_input_count', 0) > 0)
    https  = sum(1 for e in error_list if not e['features'].get('url.scheme_is_https', 0))
    subdm  = sum(1 for e in error_list if e['features'].get('url.subdomain_length', 0) > 15)
    hidden = sum(1 for e in error_list if e['features'].get('html.hidden_input_count', 0) > 2)
    total  = len(error_list)
    print(f'      No navigation : {nav}/{total} ({nav/total*100:.0f}%)')
    print(f'      No footer     : {footer}/{total} ({footer/total*100:.0f}%)')
    print(f'      Has password  : {passwd}/{total} ({passwd/total*100:.0f}%)')
    print(f'      No HTTPS      : {https}/{total} ({https/total*100:.0f}%)')
    print(f'      Long subdomain: {subdm}/{total} ({subdm/total*100:.0f}%)')
    print(f'      Hidden inputs : {hidden}/{total} ({hidden/total*100:.0f}%)')

summarize_error_features(fp_errors, 'FP')
summarize_error_features(fn_errors, 'FN')


# ============================================================
# SAMPLE ERRORS with model output signals
# ============================================================
if errors:
    print(f'\n--- SAMPLE ERRORS (first 5) ---')
    for i, e in enumerate(errors[:5]):
        f = e['features']
        model_p, model_b = extract_model_signals(e['output'])
        gt_p, gt_b       = build_signals(f)
        print(f'\n{i+1}. TRUE={e["true"].upper()} | PREDICTED={e["predicted"].upper()}')
        print(f'   URL: {e["url"][:80]}')
        print(f'   GT phishing signals  : {len(gt_p)} | GT benign signals: {len(gt_b)}')
        print(f'   Model phishing output: {len(model_p)} | Model benign output: {len(model_b)}')
        print(f'   Model output (first 200 chars): {e["output"][:200]}')
        print('-'*60)


# ============================================================
# SAMPLE AMBIGUOUS
# ============================================================
if ambiguous_examples:
    print(f'\n--- AMBIGUOUS ({len(ambiguous_examples)}) ---')
    for i, ex in enumerate(ambiguous_examples[:3]):
        print(f'\n{i+1}. [{ex["true_label"].upper()}] {ex["url"][:80]}')
        print(f'Output:\n{ex["output"][:300]}')
        print('-'*60)